# Bring your own CNN: a trained U-Net from checkpoint to Earth Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)

Runs a U-Net you trained with
[segmentation-models-pytorch](https://github.com/qubvel-org/segmentation_models.pytorch) (SMP)
inside Earth Engine, on your own imagery.

**How to use it**

- **One setting decides how far a run goes:** `RUN_UP_TO`, in step 2. **Runtime → Run all**
  goes up to that step and skips the steps already done, so you can move it one step further
  and run all again.
- **Start offline**, as shipped: a small demo U-Net with random weights. Nothing is sent to
  Earth Engine. Each cell prints what it did.
- **Try the Earth Engine steps with the demo:** set `PROJECT`, then move `RUN_UP_TO` on. The
  demo reads a public Sentinel-1 scene at Sun Moon Lake, Taiwan, with small 64-pixel tiles.
  Its random weights make a meaningless map: the point is to see each step work, and Earth
  Engine agree with PyTorch.
- **Then your model:** set `CHECKPOINT`, your model's settings, `INPUT_IMAGE` and the region,
  and start again from `"sign in"`.

| `RUN_UP_TO` | adds | creates in your project |
|---|---|---|
| `"offline"` | everything, without Earth Engine | nothing |
| `"sign in"` | signs in and sets up the prediction | nothing |
| `"upload weights"` | step 7: uploads the weights | table assets (3 for the demo) |
| `"export test tile"` | step 9: exports one test tile | one image asset |
| `"compare with PyTorch"` | step 10: compares that tile with PyTorch | nothing |
| `"export region"` | step 11: exports the whole region | one image asset |

**Cost:** the Earth Engine steps use your project's Earth Engine quota (EECU time). That is
free on a noncommercial project; a commercial project pays its plan's rate. Their cost has not
been measured for this notebook.

**What has been checked:** TensorEE's building blocks (its versions of each layer) agree with
PyTorch at their validated settings, and one trained ResNet-18 U-Net, prepared this way,
matched PyTorch to 1.2e-5 in Earth Engine on a 64 × 64 input. This tiled recipe has not been
run live as a whole: step 10 checks it for your model. Agreement with PyTorch is not accuracy.

In [ ]:
# @title 1. Install
# Installs from the repository, which gets support for more layers between releases.
!pip install --quiet "tensor-ee[segwater] @ git+https://github.com/noelEOS/tensor-ee"
print("TensorEE is installed.")

## 2. Settings

- Each line says whether you need to change it. As shipped, the notebook runs offline with the
  demo model, and none of them need changing.
- Your input image must already be in Earth Engine, with your model's bands, scaled as in
  training. It must cover the region plus a border on every side (step 6 prints the exact area).

In [ ]:
# How far this run goes. Each Run all goes up to this step and skips the steps already done.
RUN_UP_TO = "offline"  # @param ["offline", "sign in", "upload weights", "export test tile", "compare with PyTorch", "export region"]
PROJECT = "YOUR_EE_PROJECT"   # change to your Cloud project ID, as in the Code Editor

# Your model: must match how it was trained
CHECKPOINT = None             # leave None for the demo model; or your saved state_dict, for
                              # example "my_unet.pth" (in Colab, upload it in the Files panel)
ENCODER = "resnet34"          # change to the encoder_name you trained with
BANDS = ("VV", "VH")          # change to your input bands, in training order
CLASSES = 2                   # change to your number of classes

# Your input image and region (the defaults are the demo's: Sun Moon Lake, Taiwan)
INPUT_IMAGE = "COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240103T100134_20240103T100159_051941_06469E_B0BD"
                                       # change to your image; the default is a public Sentinel-1 scene
CRS = "EPSG:32651"                     # change to a projection in metres, such as your UTM zone
REGION_X, REGION_Y = 286030, 2640860   # change to the region's upper-left corner, in CRS metres
REGION_COLUMNS, REGION_ROWS = 2, 1     # change to the region's size, in tiles of TILE pixels

# Sizes: leave as they are unless your pixels are not 10 m or an export runs out of memory
PIXEL = 10                    # metres per pixel
TILE = 1024                   # side of each region tile, in pixels; larger tiles waste less on borders
TEST_TILE = 256               # side of the test tile, in pixels

# Names for what the Earth Engine steps create (the demo uses its own, starting tensor_ee_demo_).
# An existing asset is reused, not overwritten: use new names when you change the model or region.
WEIGHTS_RECORD = "my_unet_weights.json"   # local file listing your uploaded weight tables: keep it
TEST_ASSET = "my_unet_test_tile"          # image asset for the test tile
REGION_ASSET = "my_unet_region"           # image asset for the region

model_name = CHECKPOINT or "the demo model (random weights)"
if RUN_UP_TO == "offline":
    print(f"Settings saved. Model: {model_name}. Offline: nothing will be sent to Earth Engine.")
else:
    print(f'Settings saved. Model: {model_name}. This run goes up to "{RUN_UP_TO}" in project {PROJECT}.')

In [ ]:
# @title 3. Connect to Earth Engine, or stay offline
# @markdown Offline, the notebook prepares every Earth Engine request but sends none. You can
# @markdown change `RUN_UP_TO` at any time and run all again.
import ee
from ee.apitestcase import ApiTestCase

STAGES = ("offline", "sign in", "upload weights", "export test tile", "compare with PyTorch",
          "export region")


class Stop(Exception):
    """Stops the notebook with a one-line message instead of a traceback."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


def live_step(stage):
    """True when this run goes as far as `stage`; otherwise says why the step does not run."""
    if RUN_UP_TO == "offline":
        print('Not run: the notebook is offline (RUN_UP_TO is "offline" in step 2).')
        return False
    if STAGES.index(RUN_UP_TO) < STAGES.index(stage):
        print(f'Not run: set RUN_UP_TO to "{stage}" in step 2 to go this far.')
        return False
    if not PROJECT or PROJECT == "YOUR_EE_PROJECT":
        raise Stop("set PROJECT to your Cloud project ID in step 2.")
    return True


if RUN_UP_TO not in STAGES:
    raise Stop(f"RUN_UP_TO must be one of: {', '.join(STAGES)}.")
# Offline, Earth Engine's own test helper stands in for a signed-in session.
if "offline" in globals():    # undo an earlier offline run in this session
    offline.tearDown()
    ee.data._install_cloud_api_resource = _install_cloud_api_resource
    ee.Reset()
    del offline
if RUN_UP_TO == "offline":
    _install_cloud_api_resource = ee.data._install_cloud_api_resource
    offline = ApiTestCase()
    offline.setUp()
    print("Offline mode: nothing will be sent to Earth Engine or created in a project.")
else:
    print("Earth Engine mode: the next cell signs you in.")

In [ ]:
# @title Sign in
# @markdown The first time, follow the link with the Google account you use for Earth Engine.
if live_step("sign in"):
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print(f"Earth Engine is ready — signed in to {PROJECT}.")

In [ ]:
# @title 4. Load the model
# @markdown - Without a `CHECKPOINT`: a small demo U-Net with random weights, which reads
# @markdown   Sentinel-1 VV and VH in dB, rescaled to about −1 to 1 as (dB + 20) / 20.
# @markdown - Your checkpoint must match `ENCODER`, `BANDS` and `CLASSES`: if not, this cell fails
# @markdown   with an error listing the weight names that do not fit. Its input is used as it is.
import torch
import segmentation_models_pytorch as smp

torch.set_num_threads(2)


def make_model(**changes):
    settings = dict(encoder_name=ENCODER, encoder_weights=None, in_channels=len(BANDS),
                    classes=CLASSES, activation=None)
    settings.update(changes)
    return smp.Unet(**settings).eval()


if CHECKPOINT is None:
    torch.manual_seed(20261020)
    model = make_model(encoder_depth=3, decoder_channels=(16, 8, 4))
    tile, test_tile = 64, 64      # small tiles keep the demo cheap
    WEIGHTS_RECORD = "tensor_ee_demo_weights.json"
    TEST_ASSET, REGION_ASSET = "tensor_ee_demo_test_tile", "tensor_ee_demo_region"
    print("Loaded the demo model: a small U-Net with random weights.")
else:
    model = make_model()
    model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu", weights_only=True), strict=True)
    tile, test_tile = TILE, TEST_TILE
    print(f"Loaded your model from {CHECKPOINT}.")

In [ ]:
# @title 5. Check the model, its border and its weights
# @markdown - Checks that TensorEE has a building block for every layer of the model.
# @markdown - Each tile is computed from the tile plus an extra border of input on every side
# @markdown   (the halo), so that its edges come out as they would on the whole image; the
# @markdown   border is then dropped. For a ResNet-34 U-Net it is 544 pixels, so a 1,024-pixel
# @markdown   tile reads 2,112 × 2,112.
# @markdown - Counts the weights step 7 uploads. BatchNorm is first merged into the convolutions.
import math

from tensor_ee import check_compatibility, effective_convolutions
from tensor_ee.receptive_field import analyze_receptive_field
from tensor_ee.smp import adapt_smp_model


def prepare(height, width):
    """A copy of the model for one input size; stops unless TensorEE can build every layer."""
    shape = (1, len(BANDS), height, width)
    adapted = adapt_smp_model(model, input_shape=shape)
    report = check_compatibility(adapted, shape)
    if report.status != "supported":
        raise Stop(f"TensorEE cannot run this model yet: {report.to_dict()['blockers'][:3]} "
                   "(notebook 02 explains why a model is refused).")
    return adapted


probe = prepare(64, 64)
support = analyze_receptive_field(probe)
halo, alignment = support.safe_halo, support.alignment
parameters = effective_convolutions(probe)
values = sum(p.weights.size + (0 if p.bias is None else p.bias.size) for p in parameters)
tables = math.ceil(values * 14 / 7_000_000)

print("Supported: TensorEE has a building block for every layer of this model.")
if probe.tensor_ee_smp_metadata["torch_equal"]:
    print("The copy prepared for Earth Engine gives exactly your model's output on a test input.")
print(f"Border (halo): {halo} px ({halo * PIXEL / 1000:g} km) on each side. "
      f"Tile sides must be multiples of {alignment} px.")
print(f"Weights to upload: {values:,} numbers, about {tables} table asset(s).")

In [ ]:
# @title 6. Plan the tiles
# @markdown The test tile is the region's upper-left tile. Each tile reads its border of real input.
from tensor_ee import Grid
from tensor_ee.converter import PixelWindow, plan_tiles


def plan_region(side, columns, rows):
    for name, value in (("tile side", side), ("border", halo)):
        if value % alignment:
            raise Stop(f"the {name} ({value} px) must be a multiple of {alignment} px.")
    padded = Grid(CRS, (PIXEL, 0, REGION_X - halo * PIXEL, 0, -PIXEL, REGION_Y + halo * PIXEL),
                  columns * side + 2 * halo, rows * side + 2 * halo)
    region = PixelWindow(halo, halo, columns * side, rows * side)
    return plan_tiles(padded, side, halo, alignment=alignment, roi=region)


def describe(plan, name):
    context, side = plan.tiles[0].context, plan.core_size[0]
    ratio = sum(t.context.pixels for t in plan.tiles) / plan.roi.pixels
    print(f"{name}: {len(plan.tiles)} tile(s) of {side} × {side} px, each reading "
          f"{context.width} × {context.height} px with its border "
          f"({ratio:.1f} input pixels per output pixel).")


test_plan = plan_region(test_tile, 1, 1)
region_plan = plan_region(tile, REGION_COLUMNS, REGION_ROWS)
describe(test_plan, "Test tile")
describe(region_plan, "Region")
needed = region_plan.global_grid
print(f"Your input image must cover {needed.width} × {needed.height} px from the upper-left "
      f"corner ({needed.transform[2]}, {needed.transform[5]}) in {CRS}.")
if CHECKPOINT is None:
    print("(The demo uses 64 px tiles, not TILE and TEST_TILE.)")

In [ ]:
# @title 7. Upload the weights (once)
# @markdown - **This takes a while: about 7 minutes for the demo, longer for a bigger model.**
# @markdown   The bar below updates every 15 seconds. Earth Engine reports each table only as
# @markdown   waiting, running or done, so the bar moves one table at a time.
# @markdown - Creates the weight table assets, named `projects/PROJECT/assets/tensor_ee_weights_<random>_part_0000`
# @markdown   and so on (`tensor_ee_demo_weights_…` for the demo), with one export task each, two at a time.
# @markdown - Writes `WEIGHTS_RECORD`, the local list of those tables, which the prediction reads.
# @markdown   Keep a copy: in Colab, download it from the Files panel, since Colab's files are
# @markdown   lost when its runtime is reset.
# @markdown - Skipped when that file shows a finished upload; an upload that stopped part way carries on.
# @markdown - Cost not measured. For scale, a trained ResNet-18 U-Net (14.3 M numbers) took 29 tables.
import json
import time
import uuid
from pathlib import Path

from tensor_ee import export_weights

if live_step("upload weights"):
    record_path = Path(WEIGHTS_RECORD)
    record = json.loads(record_path.read_text()) if record_path.exists() else None
    if record and record["status"] == "ready":
        print(f"Already uploaded: {record_path} lists the weight tables, so this step is skipped.")
    else:
        if record and record["status"] != "uploading":
            raise Stop(f"an earlier upload {record['status']} (see {record_path}). Delete that file, "
                       "and its tables in the Code Editor's Assets tab, then run again.")
        if record:
            namespace = record["asset_namespace"]
            print("Carrying on with the upload that stopped part way.", flush=True)
        else:
            prefix = "tensor_ee_demo_weights" if CHECKPOINT is None else "tensor_ee_weights"
            namespace = f"projects/{PROJECT}/assets/{prefix}_{uuid.uuid4().hex[:12]}"
            print("Uploading; follow it here or in the Tasks tab: "
                  "https://code.earthengine.google.com/tasks", flush=True)
        upload_began = time.time()

        def progress(record):
            states = [shard["status"] for shard in record["shards"]]
            done, running = states.count("COMPLETED"), states.count("RUNNING")
            total = max(tables, len(states))
            minutes, seconds = divmod(int(time.time() - upload_began), 60)
            line = (f"  [{'█' * done}{'░' * (total - done)}] {done} of about {total} tables done, "
                    f"{running} running, {minutes} min {seconds:02d} s")
            print(line.ljust(80), end="\r", flush=True)

        export_weights(parameters, namespace, record_path, on_update=progress, resume=record is not None)
        print()
        print(f"Done: the weights are uploaded, and {record_path} lists them.")

In [ ]:
# @title 8. Set up the prediction (nothing runs yet)
# @markdown - Connects the model to your input image and uploaded weights, for the test tile.
# @markdown   Until the weights are uploaded, placeholder weights stand in: the prediction is
# @markdown   complete but cannot run.
# @markdown - Output bands: `score_k` (the model's raw scores, or logits), `p_k` (probabilities),
# @markdown   `class_index` and, for two classes, `binary_class` (1 where `p_1` ≥ 0.5).
# @markdown - Pixels missing any input band get no prediction.
import json
from pathlib import Path

from tensor_ee import MatchedWeightProvider, Raster, binary_threshold, segmentation_outputs
from tensor_ee.converter import compile_tiled_model
from tensor_ee.converter.request import deep_recursion

SCORES = tuple(f"score_{k}" for k in range(CLASSES))
PROBABILITIES = tuple(f"p_{k}" for k in range(CLASSES))


def weights_record():
    """The uploaded weight tables if the record shows a finished upload, else placeholders."""
    if RUN_UP_TO != "offline" and Path(WEIGHTS_RECORD).exists():
        record = json.loads(Path(WEIGHTS_RECORD).read_text())
        if record["status"] == "ready":
            return record
    return {"format": "native_ee_featurecollection_weights_v2", "status": "ready",
            "layers": {p.key: {"shape": list(p.shape), "dtype": p.dtype, "bias": p.bias is not None,
                               "source": p.source, "sha256": p.sha256,
                               "assets": ["projects/YOUR_EE_PROJECT/assets/weights_not_uploaded"]}
                       for p in parameters}}


def input_image():
    """The model's input in Earth Engine (a blank stand-in offline)."""
    if RUN_UP_TO == "offline":
        return ee.Image.constant([0] * len(BANDS)).rename(list(BANDS))
    image = ee.Image(INPUT_IMAGE).select(list(BANDS))
    if CHECKPOINT is None:
        image = image.add(20).divide(20)   # the demo's input: dB to about -1 to 1
    return image


def build_prediction(plan):
    context = plan.tiles[0].context
    adapted = prepare(context.height, context.width)
    if analyze_receptive_field(adapted).safe_halo != halo:
        raise Stop("the border changed with the input size; please report it as an issue.")
    provider = MatchedWeightProvider(adapted, weights_record())
    source = input_image().select(list(BANDS)).toFloat().reproject(plan.global_grid.projection())
    valid = Raster(source.mask().reduce(ee.Reducer.min()).rename("valid"), ("valid",), plan.global_grid)
    logits = compile_tiled_model(adapted, source, BANDS, plan, weight_provider=provider)
    outputs = segmentation_outputs(logits, logit_bands=SCORES, probability_bands=PROBABILITIES,
                                   validity=valid)
    bands = [outputs.logits.image, outputs.probabilities.image, outputs.classes.image]
    if CLASSES == 2:
        bands.append(binary_threshold(outputs.probabilities, positive_class=1, threshold=0.5,
                                      output_band="binary_class").image)
    return ee.Image.cat(bands).toFloat()   # one type for export; class values stay exact


def export_task(image, plan, asset_name, description):
    grid = plan.roi_grid
    return ee.batch.Export.image.toAsset(
        image=image, description=description, assetId=f"projects/{PROJECT}/assets/{asset_name}",
        region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
        maxPixels=grid.width * grid.height, pyramidingPolicy={".default": "sample"}, overwrite=False)


weights_uploaded = weights_record().get("asset_namespace") is not None
with deep_recursion():
    test_prediction = build_prediction(test_plan)
    test_graph_bytes = len(test_prediction.serialize().encode())   # checks it can be sent
if RUN_UP_TO == "offline":
    print("The test-tile prediction is set up offline, with placeholder weights: complete, "
          "but it cannot run.")
elif weights_uploaded:
    print("The test-tile prediction is set up with your uploaded weights.")
else:
    print('The test-tile prediction is set up with placeholder weights; "upload weights" '
          "uploads the real ones.")

In [ ]:
# @title 9. Export the test tile
# @markdown - Computes the test tile in a batch export and saves it as the image asset `TEST_ASSET`
# @markdown   (the tile only, not its border). Skipped when that asset already exists.
# @markdown - Cost not measured. For scale, notebook 01's 224 × 224 chip used about 65 EECU-seconds.
# @markdown - If it fails for lack of memory or time, lower `TEST_TILE` in step 2 and run all again.
import time

TASKS_URL = "https://code.earthengine.google.com/tasks"
if "started" not in globals():
    started = {}   # exports started in this session, by asset


def asset_exists(asset):
    try:
        ee.data.getAsset(asset)
        return True
    except ee.EEException:
        return False


def wait(task, minutes):
    """Wait for an export, with a line every 5 minutes; stop if it fails."""
    begun, polls = time.time(), 0
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        if time.time() - begun > minutes * 60:
            raise Stop(f"still running after {minutes} minutes. It carries on in Earth Engine: "
                       f"follow it at {TASKS_URL}, and run all again once it has finished.")
        time.sleep(30)
        polls += 1
        if polls % 10 == 0:
            print(f"  still running, {polls // 2} min...", flush=True)
    if state != "COMPLETED":
        raise Stop(f"the export ended {state}: {task.status().get('error_message', '')} "
                   f"(see {TASKS_URL})")


def export_once(asset_name, make_image, plan, description, minutes):
    """Export to `asset_name` unless it exists; wait for an export this session already started."""
    asset = f"projects/{PROJECT}/assets/{asset_name}"
    task = started.get(asset)
    if task is not None and task.status()["state"] not in ("COMPLETED", "FAILED", "CANCELLED"):
        print("The export started earlier is still running; waiting for it.", flush=True)
    elif asset_exists(asset):
        print(f"Already exported: {asset} exists, so this step is skipped.")
        return
    else:
        if not weights_uploaded:
            raise Stop('the weights are not uploaded yet: set RUN_UP_TO to "upload weights" or later.')
        with deep_recursion():
            task = export_task(make_image(), plan, asset_name, description)
            task.start()
        started[asset] = task
        print(f"Export started; follow it here or at {TASKS_URL}", flush=True)
    wait(task, minutes)
    print(f"Done: saved as {asset}.")


if live_step("export test tile"):
    export_once(TEST_ASSET, lambda: test_prediction, test_plan, "tensor_ee_test_tile", minutes=60)

In [ ]:
# @title 10. Compare the test tile with PyTorch
# @markdown - Downloads the test tile's input (with its border) and the exported scores: 2 small
# @markdown   requests, nothing created.
# @markdown - Runs your model here on the same input and prints the largest difference
# @markdown   (target: at most 1e-4). The input must have no gaps; above 40 MB it is refused.
import numpy as np

within_target = False


def download(image, grid, bands):
    names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
    values = ee.data.computePixels({
        "expression": image.select(list(bands)), "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": grid.width, "height": grid.height},
                 "affineTransform": dict(zip(names, grid.transform)), "crsCode": grid.crs}})
    return np.stack([values[band] for band in bands]).astype(np.float32)


if live_step("compare with PyTorch"):
    padded = test_plan.global_grid
    size = padded.width * padded.height * (len(BANDS) + 1) * 4
    if size > 40_000_000:
        raise Stop(f"the input is {size / 1e6:.0f} MB; lower TEST_TILE in step 2.")
    source = input_image().select(list(BANDS)).toFloat().reproject(padded.projection())
    valid = source.mask().reduce(ee.Reducer.min()).rename("valid")
    pixels = download(source.unmask(0).addBands(valid), padded, (*BANDS, "valid"))
    if pixels[-1].min() < 1:
        raise Stop("the input has gaps in the test tile or its border; move the region or fill them.")
    exported = download(ee.Image(f"projects/{PROJECT}/assets/{TEST_ASSET}"), test_plan.roi_grid, SCORES)

    with torch.no_grad():
        reference = model(torch.from_numpy(pixels[None, :-1].copy()))[0].numpy()
    core = reference[:, halo:halo + test_tile, halo:halo + test_tile]
    error = np.abs(exported - core)
    within_target = bool(error.max() <= 1e-4)
    print(f"Largest difference in the scores: {error.max():.3e} (target 1e-4); "
          f"mean {error.mean():.3e}; pixels with a different class: "
          f"{int((exported.argmax(0) != core.argmax(0)).sum())} of {core[0].size}.")
    print("Done: within the target. Earth Engine agrees with PyTorch on this tile." if within_target
          else "Done, but above the target: please report it as an issue, with these numbers.")
    if CHECKPOINT is None:
        print("(The demo's random weights make a meaningless map: this shows only that Earth "
              "Engine computes the same numbers as PyTorch.)")

In [ ]:
# @title 11. Export the region
# @markdown - One batch export, saved as the image asset `REGION_ASSET` (the region only, not its
# @markdown   border). Skipped when that asset already exists.
# @markdown - Runs only when step 10 was within the target in this run.
# @markdown - Cost, memory use and time not measured at these tile sizes.
if live_step("export region"):
    if not within_target:
        raise Stop("step 10 was not within the target, so the region is not exported.")
    export_once(REGION_ASSET, lambda: build_prediction(region_plan), region_plan,
                "tensor_ee_region", minutes=240)
    if CHECKPOINT is None:
        print("(The demo's random weights make a meaningless map: open it only to see that the "
              "export worked.)")

## Good to know

- **Coming back later:** put your `WEIGHTS_RECORD` file back next to the notebook (in Colab,
  upload it in the Files panel, like the checkpoint), and **Run all**: the finished steps are
  skipped.
- **Evidence:** step 10 is agreement in Earth Engine for your model on one tile; keep its
  numbers. Accuracy is separate: evaluate the exports against your own labelled data.
- **Details:** [the guide this notebook follows](https://github.com/noelEOS/tensor-ee/blob/main/docs/own-model.md)
  covers other encoders, irregular regions and masks of valid pixels.
- **Clean up:** the weight tables and images stay in your project until you delete them (Code
  Editor, **Assets** tab). Keep the weight tables while you predict with them.